####clicked_items → Array of Array

#### ordered_products → Array of Struct

####promo_info → Array of Struct

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
from common_utils.scd_functions import scd1
from common_utils.logging import get_logger

logger = get_logger("sales_orders_silver")


df = spark.read.table("retaildataplatform.bronze.cosmosdb_sales_orders")


#Important: This schema assumes your Bronze column is json_data, as in your class example. If your Bronze table already has these fields parsed into separate columns, you don't need from_json().

json_schema = StructType([
    StructField("_id", StructType([
        StructField("$oid", StringType(), True)
    ]), True),

#clicked_items → Array of Array
    StructField("clicked_items", ArrayType(
        ArrayType(StringType())
    ), True),

    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("number_of_line_items", IntegerType(), True),
    StructField("order_datetime", DoubleType(), True),
    StructField("order_number", LongType(), True),

# ordered_products → Array of Struct
    StructField("ordered_products", ArrayType(
        StructType([
            StructField("curr", StringType(), True),
            StructField("id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("price", StringType(), True),
            StructField("promotion_info", StringType(), True),
            StructField("qty", StringType(), True),
            StructField("unit", StringType(), True)
        ])
    ), True),

# promo_info → Array of Struct
    StructField("promo_info", ArrayType(
        StructType([
            StructField("promo_disc", DoubleType(), True),
            StructField("promo_id", StringType(), True),
            StructField("promo_item", StringType(), True),
            StructField("promo_qty", StringType(), True)
        ])
    ), True)
])

df_parsed = df.withColumn(
    "data",
    F.from_json(
        F.col("json_data"),
        json_schema
    )
)

def transform_sales_orders(df):
    return df.select(
        F.col("data.order_number").alias("order_number"),
        F.col("data.customer_id").alias("customer_id"),
        F.col("data.customer_name").alias("customer_name"),
        F.col("data.number_of_line_items").alias("number_of_line_items"),
        F.from_unixtime( F.col("data.order_datetime")).alias("order_timestamp")
    )

def transform_order_products(df):
    return df.select(
        F.col("data.customer_id").alias("customer_id"),
        F.col("data.customer_name").alias("customer_name"),
        F.col("data.order_number").alias("order_number"),
        F.explode(F.col("data.ordered_products")).alias("product")
    ).select(
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("order_number"),
        F.col("product.id").alias("id"),
        F.col("product.name").alias("product_name"),
        F.col("product.price").alias("price"),
        F.col("product.curr").alias("curr"),
        F.col("product.qty").alias("qty"),
        F.col("product.unit").alias("unit")
    )

def transform_promotions(df):
    return df.select(
        F.col("data.customer_id").alias("customer_id"),
        F.col("data.order_number").alias("order_number"),
        F.explode(F.col("data.promo_info")).alias("promo")
    ).select(
        F.col("customer_id"),
        F.col("order_number"),
        F.col("promo.promo_id").alias("promo_id"),
        F.col("promo.promo_item").alias("promo_product_id"),
        F.col("promo.promo_disc").alias("discount"),
        F.col("promo.promo_qty").cast("int").alias("promo_quantity")
    )




def transform_clicked_items(df):
    return df.select(
        F.col("data.customer_id").alias("customer_id"),
        F.col("data.customer_name").alias("customer_name"),
        F.col("data.order_number").alias("order_number"),
        F.explode(F.col("data.clicked_items")).alias("clicked")
    ).select(
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("order_number"),
        F.col("clicked")[0].alias("product_id"),
        F.col("clicked")[1].alias("score")
    )


sales_orders_df = transform_sales_orders(df_parsed)
sales_orders_df = sales_orders_df.dropDuplicates()

order_products_df = transform_order_products(df_parsed)
order_products_df = order_products_df.dropDuplicates()

promotions_df = transform_promotions(df_parsed)
promotions_df = promotions_df.dropDuplicates()

clicked_items_df = transform_clicked_items(df_parsed)
clicked_items_df = clicked_items_df.dropDuplicates()


scd1(
    spark,
    sales_orders_df,
    ["order_number", "customer_id", "number_of_line_items"],
    "retaildataplatform.silver.sales_orders"
)

scd1(
    spark,
    order_products_df,
    ["order_number", "id", "price"],
    "retaildataplatform.silver.order_products"
)

scd1(
    spark,
    promotions_df,
    ["order_number", "promo_product_id", "promo_quantity"],
    "retaildataplatform.silver.promotions"
)

scd1(
    spark,
    clicked_items_df,
    ["order_number", "product_id", "score"],
    "retaildataplatform.silver.clicked_items"
)

logger.info("All sales order entities successfully processed into Silver")

In [0]:
display(
    spark.table("retaildataplatform.silver.clicked_items")
)